In [ ]:
import pandas as p
import numpy as np
!pip install mendeleev==0.9.0 --quiet
from mendeleev import element
import matplotlib.pyplot as plt
from itertools import product
import cmath

In [ ]:
car = {
  "brand": "Ford",
  "model": "Mustang",
  "year": 1964
}

x = car.keys()

print(x)

In [ ]:
!jupyter notebook --generate-config

In [ ]:
def factorial(n):
  return 1 if n == 0 else n * factorial(n-1)
#
#
#DEFINE UNIT LATTICE PARAMS, MOTIF
use_preset = input("Use preset? (y/n)")
if use_preset == "y":

  motif_elements = ["Ag","Ag","Ag","Ag","Ag","Ag","Ag","Ag","Ag","Ag","Ag","Ag","Ag","Ag","Ag","Cl","Cl","Cl","Cl","Cl","Cl","Cl","Cl","Cl","Cl","Cl","Cl"]
  motif_cell_coord = [[0,0,0],[1,0,0],[0,1,0],[1,1,0],[0.5,0.5,0],[0.5,0,0.5],[0,0.5,0.5],[1,0.5,0.5],[0.5,1,0.5],[0,0,1],[1,0,1],[0,1,1],[1,1,1],[0.5,0.5,1],[0.5,0,0],[0,0.5,0],[1,0.5,0],[0.5,1,0],[0.5,0,1],[0,0.5,1],[1,0.5,1],[0.5,1,1],[0,0,0.5],[1,0,0.5],[0,1,0.5],[1,1,0.5],[0.5,0.5,0.5]]

  motif_f = []
  for i in motif_elements:
    motif_f.append(element(i).atomic_number)

  a_value = 5.55
  b_value = 5.55
  c_value = 5.55
  alpha = 90
  beta = 90
  gamma = 90
  wavelength = 0.15405
else:
  #define lattice parameters
  a_value = float(input("[Å] a = "))
  b_value = float(input("[Å] b = "))
  c_value = float(input("[Å] c = "))
  alpha = float(input("[°] α = "))
  beta = float(input("[°] β = "))
  gamma = float(input("[°] γ = "))
  wavelength = 10 * float(input("[nm] λ = "))


  #df headings, loop setup
  print("Only input integer or fractional coordinates (no decimals)")
  motif_elements = []
  motif_cell_coord = []
  motif_f = []
  user_continue = "y"

  #def user input -> float
  def fractionise(x):
    if '/' in x:
      num, den = x.split('/')
      return float(num)/float(den)
    else:
      return float(x)

  #user input motif
  while user_continue == "y":
    motif_element = input("element = ")
    motif_elements.append(motif_element)

    motif_coordinate_x = fractionise(input("x = "))
    motif_coordinate_y = fractionise(input("y = "))
    motif_coordinate_z = fractionise(input("z = "))
    motif_coordinate = [motif_coordinate_x, motif_coordinate_y, motif_coordinate_z]
    motif_cell_coord.append(motif_coordinate) #using abc basis

    user_continue = input("add another atom? (y/n) ")
    motif_f.append(element(motif_element).atomic_number)



#coordinates: unit cell basis <-> xyz basis
#convert unit cell basis -> abc (in xyz basis)
a = np.array([a_value,0,0])
b = np.array([b_value*np.cos(np.deg2rad(gamma)),b_value*np.sin(np.deg2rad(gamma)),0])
c_x = float(c_value*np.cos(np.deg2rad(gamma)))
c_y = float(b_value*np.cos(np.deg2rad(alpha))*np.cos(np.pi / 2 - np.deg2rad(beta)))
c = np.array([c_x,c_y,np.sqrt(c_value ** 2 - c_x ** 2 - c_y ** 2)]) #look at grasshopper code

#motif_cell_coord_cart = (np.array(motif_cell_coord) * np.array([a,b,c])).tolist() #need this?

# UP change this to matrix form: inv.array(a,b,c)*(coord) = cart coord

abc = np.array([a,b,c])
abc_inv = np.linalg.inv(abc)

motif_coord = []
for i in motif_cell_coord:
  motif_coord.append(np.dot(i,abc))

#defined:
  #wavelength
  #vectors abc (in xyz)
  #abc lengths
  #motif_elements
  #motif_cell_coord (abc)
  #motif_coord (xyz)
  #motif_f

#NEED XYZ COORDS

In [ ]:
# DEFINE FUNCTIONS, VARIABLES

#geometric factors
def g(x): #note: x should be in deg
  try:
    return (1 + np.cos(2 * np.deg2rad(x)) ** 2) / (np.sin(np.deg2rad(x)) * np.cos(np.deg2rad(x)))
  except ZeroDivisionError:
    print("error")
    return 1

#test if float is in an array
def float_in_array(value, array, tol):
  for i in array:
    if abs(i - value) < tol:
      return True
  return False

def d(h,k,l):
  return 1/np.sqrt((h/a_value)**2 + (k/b_value)**2 + (l/c_value)**2)

def zerodiv(a,b):
    if b == 0:
      return 0
    else:
      return a/b

def normal(hkl):

    h = hkl[0]
    k = hkl[1]
    l = hkl[2]

    vec_ind = [ [a,h], [b,k], [c,l] ]
    vec_in_plane = []

    for i in vec_ind:
      if i[1] == 0:
        vec_in_plane.append(i[0])


    if [h,k,l].count(0) == 0:
      try:
        normal = np.cross( (zerodiv(a,h) - zerodiv(b,k)) , (zerodiv(a,h) - zerodiv(c,l)) ) ### may cause error (crossing 0 -- not array)
      except:
        normal = [0,0,0]


    if [h,k,l].count(0) == 1:
      vec_in_plane = []
      vec_to_subtract = []
      for i in vec_ind:
        if i[1] == 0:
          vec_in_plane.append(i[0])
        else:
          vec_to_subtract.append(i[0]/i[1])

      #print(f"vec_in_plane = {vec_in_plane}, vec_to_subtract = {vec_to_subtract}")

      vec_in_plane.append(vec_to_subtract[0] - vec_to_subtract[1])
      normal = np.cross(vec_in_plane[0], vec_in_plane[1])

    if [h,k,l].count(0) == 2:
      for i in vec_ind:
        if i[1] != 0:
          normal = i[0]

    #normal[np.isclose(normal, 0, atol=1e-4)] = 0

    return normal

def atom_in_plane(coord,hkl):

  x = coord[0]
  y = coord[1]
  z = coord[2]
  h = hkl[0]
  k = hkl[1]
  l = hkl[2]

  dis = d(h,k,l)

  n = np.array(normal(hkl))
  n_d = n * dis / np.linalg.norm(n)
  k = np.dot(n_d,n_d)

  #print(f"n = {n}, d = {dis}, n_d = {n_d}, k = {k}, k_coord = {x*n_d[0] + y*n_d[1] + z*n_d[2]}, k_coord / k = {((x*n_d[0] + y*n_d[1] + z*n_d[2]) / k, 5)}")
  return round((x*n_d[0] + y*n_d[1] + z*n_d[2]) / k, 5) % 1 == 0

def m(h,k,l):
  list = [h,k,l]
  list.sort()
  zeros = list.count(0)
  h = list[0]
  k = list[1]
  l = list[2]
  #no_same counts nonzero repetitions
  if h == k:
    no_same = list.count(h)
  elif k == l:
    no_same = list.count(k)
  else:
    no_same = 0
  return (6 * 8)/(2**zeros * factorial(no_same))

def f_2(h,k,l,f,cell_coord):
  complex_component = cmath.exp( complex(0, 4*np.pi * (h*cell_coord[0] + k*cell_coord[1] + l*cell_coord[2])) )
  return f**2 * complex_component * np.conjugate(complex_component)
#def F^2(h,k,l,theta):
 # return    cmath.exp(4 * d(h,k,l) * np.sin(np.deg2rad(theta)) * np.pi / wavelength) ########## * f_j and then sum for all atoms j
  ########### or: hkl pi thingy


#defined:
  #theta (array)
  # g(x)
  # d(h,k,l)
  # m(h,k,l)
  # atom_in_plane
  # f_2(h,k,l,f,cell_coord) (scattering factor, squared)
  # float in array test

In [ ]:
#ATTEMPT 2 -- select planes using Bragg's law

#theta values
theta_div = 200 #= number of theta values to test
theta = list(np.linspace(0.5,80,theta_div))

I = []

for i in theta:
  #theta fixed
  #calculate d_min
  d_min = wavelength / (2 * np.sin(np.deg2rad(i)))

  #test k multiples of d_min -> d_values to be tested calculated (reflection confirmed)
  k = 5
  d_values = []
  for j in np.arange(1,k,1):
    d_values.append(d_min*j)

        # for hkl < k2, if d(hkl) exists in d, append; else ignore

  #find hkl values which satisfy d
  hkl_max = 20
  hkl_perms = np.array(list(product(np.arange(0,hkl_max,1), repeat = 3))) #list of arrays #change to remove hkl reorderings, if works add reorderings then test if atoms in plane
  hkl_perms = hkl_perms[~np.all(hkl_perms == [0, 0, 0], axis=1)]
  allowed_hkl = [] #array for allowed hkl values (allowed by Bragg's law)
  allowed_hkl_d = [] #array for associated d values
  d_tolerance = 0.00001 #tolerance for d value to test if plane allowed

  #test if hkl spacing is valid
  for item in hkl_perms:
    h = item[0]
    k = item[1]
    l = item[2]
    spacing = d(h,k,l) #CHECK FOR WHICH UNIT CELL TYPES THIS IS VALID
    if float_in_array(spacing, d_values,d_tolerance):
      allowed_hkl.append(item)
      allowed_hkl_d.append(spacing)

       #once some hkl values found with max index 20, run this
       #while d(hkl)>min(d), append hkl s.t. d(newhkl) = 1/2d something like this!

  #test if atoms exist in plane
  #THIS CODE IS INCORRECT

  hkl_final = []
  f_final = []
  F = 0
  for loop1 in allowed_hkl:
    hkl = loop1 #3x1 array

    for loop2 in range(len(motif_coord)): #for each atom:
      coord = motif_coord[loop2]
      f = motif_f[loop2]

      if atom_in_plane(coord,hkl) == True:
        hkl_final.append(loop1)
        f_final.append(f)
        F += f_2(hkl[0],hkl[1],hkl[2],f,motif_cell_coord[loop2])

  if F != 0:
    I.append(m(hkl[0],hkl[1],hkl[2]) * F * g(i) / d(hkl[0],hkl[1],hkl[2])) #d(hkl)?
  else:
    I.append(0)

  print(f"theta = {round(i,3)}: {allowed_hkl} \n d_min = {d_min} \nhkl_final = {hkl_final} \nf_final = {f_final} \nF = {F} \n")

plt.plot(theta, I)



In [ ]:
print(allowed)

In [ ]:

#
#
#STRUCTURE FACTOR CONSTRAINS AND FORMUALE

def I_plane(h,k,l,theta):
  F = 0
  no_atoms = len(motif_df["element"].tolist())
  for i in range(no_atoms):
    coord = motif_df["coordinates"].tolist()[i]
    F_contr = cmath.exp(-2 * np.pi * (h*coord[0] +k*coord[1] +l*coord[2])) # * f FOR ATOM
    F =+ F_contr
  return g(theta)

#generate planes and data
hkl_max = 30
hkl_values = np.arange(0,hkl_max,1)

I = []
for t in theta:
  I_planes = 0
  for i in product(hkl_values, repeat = 3):
    I_planes = I_planes + (I_plane(i[0],i[1],i[2],t))
  I.append(I_planes)
  print(f"theta = ",str(round(t,3)) + "° computed")


plt.plot(theta,I)
plt.show()

In [ ]:
#max error due to ignoring planes with high d
d_err = 0.01

#test for first reflection plane, define hkl_max
hkl_max = 10
hkl_values = np.arange(0,hkl_max,1)
hkl_perms = list(product(hkl_values, repeat = 3))
hkl_spacings = []
for i in hkl_perms:
  #if has atoms
  if ((i[0]/a)**2 + (i[1]/b)**2 + (i[2]/c)**2)**(1/2) != 0:
    hkl_spacing = 1 / ((i[0]/a)**2 + (i[1]/b)**2 + (i[2]/c)**2)**(1/2)
    hkl_spacings.append(hkl_spacing)
  else:
    hkl_spacings.append(10**10) #really large instead of inf



for hkl in hkl_perms:
  #test if atoms lie in plane
  h = hkl[0]
  k = hkl[1]
  l = hkl[2]
  N = h**2 + k**2 + l**2
  d = 1 / N

  I_contribution = g(theta) * 1/d # f**2 * superposition?
  #stop iteration once higher hkl --> no more atoms in these planes



In [ ]:
import matplotlib.pyplot as plt

x = [0,1,2,3,4,5]
y = [0,1,4,9,16,25]

plt.plot(x,y)
plt.show()

In [ ]:
(F * F.conjugate()).real * m(h,k,l) * g(theta)